In [1]:
import numpy as np
from tifffile import imread
from skimage.measure import marching_cubes
import trimesh
import matplotlib.pyplot as plt

In [2]:
# ── Load annotation mask ──────────────────────────────────────────────────
DATA_DIR = '260227_ID459_laminB1antibody/anno'

mask = imread(f'{DATA_DIR}/ST28_stable_emb2_-01_nuclei_anno.tif')  # (Z, Y, X) instance labels

labels = np.unique(mask)
labels = labels[labels > 0]   # remove background

VOXEL_SIZE   = np.array([0.3, 0.14, 0.14])   # µm per voxel (Z, Y, X)
MIN_VOXELS   = 100                             # tune this — fragments are typically tiny

# filter by voxel count
sizes          = {lbl: (mask == lbl).sum() for lbl in labels}
labels_filtered = np.array([l for l in labels if sizes[l] >= MIN_VOXELS])

print(f'mask shape  : {mask.shape}')
print(f'total labels: {len(labels)}')
print(f'after filter: {len(labels_filtered)}  (removed {len(labels)-len(labels_filtered)} fragments < {MIN_VOXELS} voxels)')
print(f'size range  : {min(sizes.values())} – {max(sizes.values())} voxels')

labels = labels_filtered


mask shape  : (66, 1024, 1024)
total labels: 54
after filter: 44  (removed 10 fragments < 100 voxels)
size range  : 1 – 198075 voxels


In [3]:
# ── Build surface point cloud for each nucleus ────────────────────────────
N_POINTS = 1024   # points per nucleus (fixed for downstream ML)

def nucleus_to_pointcloud(mask, label, voxel_size, n_points):
    """Return (n_points, 3) array of surface points in µm, centered at origin."""
    binary = (mask == label).astype(np.float32)

    # marching cubes → surface mesh
    verts, faces, _, _ = marching_cubes(binary, level=0.5, spacing=voxel_size)

    # sample points uniformly from mesh surface
    mesh   = trimesh.Trimesh(vertices=verts, faces=faces)
    pts, _ = trimesh.sample.sample_surface(mesh, n_points)

    # center at origin
    pts -= pts.mean(axis=0)
    return pts

point_clouds = {}   # label → (N, 3)
for lbl in labels:
    pts = nucleus_to_pointcloud(mask, lbl, VOXEL_SIZE, N_POINTS)
    point_clouds[lbl] = pts

print(f'point clouds built: {len(point_clouds)}')
print(f'shape per nucleus : {pts.shape}')

point clouds built: 44
shape per nucleus : (1024, 3)


In [4]:
import napari

# ── Convert point clouds back to voxel space for napari ──────────────────
# point clouds are in µm centered at origin → convert to voxel coords
img = imread(f'{DATA_DIR}/ST28_stable_emb2_-01_nuclei.tif')  # (Z, Y, X)

# collect all points with a colour per nucleus
all_pts    = []
all_colors = []
cmap       = plt.cm.tab20

for i, lbl in enumerate(labels):
    pts_um = point_clouds[lbl]   # (1024, 3) in µm, centered

    # add nucleus centroid back (in voxel coords)
    zyx = np.argwhere(mask == lbl)           # (M, 3) voxel coords
    centroid_vox = zyx.mean(axis=0)          # (3,) in voxels

    # convert µm offsets back to voxels
    pts_vox = pts_um / VOXEL_SIZE + centroid_vox   # (1024, 3)
    all_pts.append(pts_vox)

    color = cmap(i % 20)
    all_colors += [color] * len(pts_vox)

all_pts    = np.vstack(all_pts)     # (N_nuclei * 1024, 3)
all_colors = np.array(all_colors)   # (N_nuclei * 1024, 4)

viewer = napari.Viewer()
viewer.add_image(img,   name='nuclei',  colormap='gray')
viewer.add_labels(mask, name='seg mask')
viewer.add_points(all_pts, face_color=all_colors, size=5, name='point cloud')
viewer.add_image(img[:50], name='nuclei copy', colormap='gray')
napari.run()


In [ ]:
# ── Save as numpy array (N_nuclei, N_points, 3) ───────────────────────────
pc_array = np.stack([point_clouds[l] for l in labels], axis=0)  # (N, 1024, 3)
np.save('point_clouds_stable_emb2.npy', pc_array)
print(f'saved: {pc_array.shape}  →  point_clouds_stable_emb2.npy')

In [14]:
print(img.shape)

(66, 1024, 1024)


In [ ]:

# ── Layer planes animation ────────────────────────────────────────────────
from napari_animation import Animation
import napari

Z_MAX = img.shape[0] - 1  # 65

viewer = napari.Viewer(ndisplay=3)

image_layer  = viewer.add_image(img,  name='nuclei',   colormap='gray',
                                depiction='plane', blending='translucent')
labels_layer = viewer.add_labels(mask, name='seg mask', blending='translucent')

viewer.camera.angles = (-15, 45, 120)
viewer.camera.zoom  *= 0.9

animation = Animation(viewer)

def update_labels_clip():
    z = int(image_layer.plane.position[0])
    new_mask = mask.copy()
    new_mask[z:] = 0
    labels_layer.data = new_mask

# ── Phase 1: plane sweeps, labels hidden ─────────────────────────────────
labels_layer.visible = False
image_layer.plane.position = (0, 0, 0)
animation.capture_keyframe(steps=120)

image_layer.plane.position = (Z_MAX, 0, 0)
animation.capture_keyframe(steps=120)

image_layer.plane.position = (0, 0, 0)
animation.capture_keyframe(steps=120)

# ── Phase 2: plane sweeps, labels revealed via clipping ───────────────────
image_layer.plane.events.position.connect(lambda e: update_labels_clip())
labels_layer.visible = True
labels_layer.experimental_clipping_planes = [{"position": (0, 0, 0), "normal": (-1, 0, 0)}]

image_layer.plane.position = (Z_MAX, 0, 0)
labels_layer.experimental_clipping_planes[0].position = (Z_MAX, 0, 0)
animation.capture_keyframe(steps=120)

image_layer.plane.position = (0, 0, 0)
labels_layer.experimental_clipping_planes[0].position = (0, 0, 0)
animation.capture_keyframe(steps=120)

animation.animate('layer_planes_nuclei.mp4', canvas_only=True,
                  fps=30, quality=9)


c:\Users\ljd567\AppData\Local\miniconda3\envs\torch_env\lib\site-packages\napari\plugins\_plugin_manager.py:555: UserWarning: Plugin 'napari_skimage_regionprops2' has already registered a function widget 'duplicate current frame' which has now been overwritten
  warn(message=warn_message)


In [ ]:

# ── Point cloud animation ─────────────────────────────────────────────────
from napari_animation import Animation
import napari

Z_MAX = img.shape[0] - 1  # 65

viewer = napari.Viewer(ndisplay=3)

image_layer  = viewer.add_image(img, name='nuclei', colormap='gray',
                                depiction='plane', blending='translucent')
points_layer = viewer.add_points(all_pts, face_color=all_colors, size=5,
                                 name='point cloud', blending='translucent')

viewer.camera.angles = (-18.23797054423494, 41.97404742075617, 141.96173085742896)
viewer.camera.zoom  *= 0.9

animation = Animation(viewer)

# ── Phase 1: plane sweeps, points hidden ─────────────────────────────────
points_layer.visible = False
image_layer.plane.position = (0, 0, 0)
animation.capture_keyframe(steps=60)

image_layer.plane.position = (Z_MAX, 0, 0)
animation.capture_keyframe(steps=60)

image_layer.plane.position = (0, 0, 0)
animation.capture_keyframe(steps=60)

# ── Phase 2: plane sweeps forward, points revealed slice by slice ─────────
# Use `shown` per z-slice instead of clipping planes (Points layers
# don't honour experimental_clipping_planes reliably)
points_layer.visible = True
for z in range(Z_MAX + 1):
    image_layer.plane.position = (z, 0, 0)
    points_layer.shown = all_pts[:, 0] <= z   # all_pts is ZYX
    animation.capture_keyframe(steps=1)

animation.animate('layer_planes_pointcloud.mp4', canvas_only=True,
                  fps=30, quality=9, scale_factor=2)


In [ ]:

# ── Load PLY point clouds with cell-select slider ─────────────────────────
from pathlib import Path
import trimesh
import napari

PLY_DIR = Path('260227_ID459_laminB1antibody/anno/ST28_stable_emb2_-01_nuclei')

all_pts  = []
ply_files = sorted(PLY_DIR.glob('*.ply'), key=lambda p: int(p.stem))
label_ids = [int(p.stem) for p in ply_files]
print(f'Found {len(ply_files)} PLY files in {PLY_DIR}')

for p in ply_files:
    pts = trimesh.load(p).vertices   # (1024, 3), centered at origin
    all_pts.append(pts)

all_pts = np.array(all_pts)   # (N_cells, 1024, 3)

# Prepend cell index as dim-0 → napari adds a slider for it
# Result shape: (N_cells * 1024, 4)  columns: [cell_idx, z, y, x]
n_cells, n_pts, _ = all_pts.shape
cell_idx = np.repeat(np.arange(n_cells), n_pts).reshape(-1, 1)
pts_4d   = np.hstack([cell_idx, all_pts.reshape(-1, 3)])

viewer = napari.Viewer(ndisplay=3)
viewer.add_points(pts_4d, size=0.3, name='cells',
                  face_color='cyan', ndim=4)
viewer.camera.angles = (-15, 45, 120)
napari.run()
